<div style="background: linear-gradient(135deg, #1a237e 0%, #283593 40%, #1565c0 100%); padding: 48px 40px; border-radius: 16px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">RSI Backtesting Engine</h1>
  <p style="color: #90caf9; font-size: 1.15em; margin: 0 0 24px 0;">Estrategia de Mean-Reversion basada en el Índice de Fuerza Relativa</p>
  <div style="display: flex; gap: 12px; flex-wrap: wrap;">
    <span style="background: rgba(255,255,255,0.15); color: white; padding: 4px 14px; border-radius: 20px; font-size: 0.85em;">📈 BTC-USD · 2020–2024</span>
    <span style="background: rgba(255,255,255,0.15); color: white; padding: 4px 14px; border-radius: 20px; font-size: 0.85em;">⚙️ RSI(14) · Oversold 30 · Overbought 70</span>
    <span style="background: rgba(255,255,255,0.15); color: white; padding: 4px 14px; border-radius: 20px; font-size: 0.85em;">💰 Capital: $10,000</span>
    <span style="background: rgba(255,255,255,0.15); color: white; padding: 4px 14px; border-radius: 20px; font-size: 0.85em;">🏦 Comisión: 0.10% · Slippage: 0.05%</span>
  </div>
</div>

## 📋 Tabla de Contenidos

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | [Configuración del entorno](#1) | Imports, logging y parámetros globales |
| 2 | [Carga y validación de datos](#2) | Descarga OHLCV, limpieza y estadísticas descriptivas |
| 3 | [El indicador RSI](#3) | Teoría, fórmula matemática e implementación vectorizada |
| 4 | [Generación de señales](#4) | Lógica de la estrategia y visualización de señales |
| 5 | [Simulación de ejecución](#5) | Broker simulator: comisiones y slippage |
| 6 | [Motor de backtesting](#6) | Ejecución completa del backtest |
| 7 | [Equity Curve & Drawdown](#7) | Evolución del capital y análisis de drawdowns |
| 8 | [Métricas de rendimiento](#8) | Sharpe, CAGR, Max DD, Win Rate y más |
| 9 | [Análisis de trades](#9) | Log detallado y estadísticas por operación |
| 10 | [Análisis de sensibilidad](#10) | Heatmap de parámetros RSI |
| 11 | [Reporte final](#11) | Dashboard completo integrado |

---
<a id='1'></a>
## 1 · Configuración del Entorno

Importamos todas las dependencias y configuramos el sistema de logging para tener visibilidad completa de la ejecución.

In [ ]:
# ── Imports estándar ────────────────────────────────────────────────────────────
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, HTML

# ── Módulos del proyecto ────────────────────────────────────────────────────────
from src.config.settings import *
from src.data.data_loader import load_ohlcv
from src.data.data_cleaner import validate_and_clean
from src.indicators.rsi import compute_rsi, add_rsi_to_df
from src.strategy.rsi_strategy import generate_signals
from src.backtester.engine import BacktestEngine
from src.metrics.performance import compute_metrics, format_metrics_table
from src.metrics.drawdown import compute_drawdown, max_drawdown
from src.utils.helpers import setup_logging

# ── Logging y estilo visual ─────────────────────────────────────────────────────
setup_logging()

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.color':       '#E0E0E0',
    'grid.linewidth':   0.6,
    'font.family':      'sans-serif',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'figure.dpi':       130,
})

# ── Paleta de colores del proyecto ─────────────────────────────────────────────
C = {
    'strategy':  '#2196F3',
    'benchmark': '#90A4AE',
    'drawdown':  '#EF5350',
    'buy':       '#26A69A',
    'sell':      '#EF5350',
    'rsi':       '#7E57C2',
    'gold':      '#FFA726',
    'bg_card':   '#F5F7FF',
}

print('✅ Entorno configurado correctamente')
print(f'   Python {sys.version.split()[0]}  |  Pandas {pd.__version__}  |  NumPy {np.__version__}')

---
<a id='2'></a>
## 2 · Carga y Validación de Datos

Descargamos datos OHLCV de Bitcoin (BTC-USD) desde Yahoo Finance para el período 2020–2024. El módulo `data_cleaner` valida la integridad: sin NaNs en precios, consistencia OHLC (High ≥ Low, etc.) y orden cronológico.

In [ ]:
# ── Descarga y limpieza ────────────────────────────────────────────────────────
raw_df   = load_ohlcv(DEFAULT_TICKER, DEFAULT_START, DEFAULT_END, DEFAULT_INTERVAL)
clean_df = validate_and_clean(raw_df)

print(f'\n📊 Dataset: {DEFAULT_TICKER}')
print(f'   Período : {clean_df.index[0].date()} → {clean_df.index[-1].date()}')
print(f'   Filas   : {len(clean_df):,}')
print(f'   Columnas: {list(clean_df.columns)}')

In [ ]:
# ── Estadísticas descriptivas ──────────────────────────────────────────────────
desc = clean_df[['Open','High','Low','Close','Volume']].describe().round(2)
display(HTML('<h4 style="color:#1565c0">Estadísticas descriptivas — BTC-USD</h4>'))
display(desc.style
    .background_gradient(cmap='Blues', subset=['Close','High','Low'])
    .format('{:,.2f}')
    .set_properties(**{'font-size': '13px'}))

In [ ]:
# ── Gráfico de precio histórico ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})

ax1, ax2 = axes

# Precio de cierre
ax1.plot(clean_df.index, clean_df['Close'], color=C['strategy'], linewidth=1.4)
ax1.fill_between(clean_df.index, clean_df['Low'], clean_df['High'],
                  alpha=0.08, color=C['strategy'])
ax1.set_title(f'{DEFAULT_TICKER} — Precio histórico ({DEFAULT_START} → {DEFAULT_END})',
               fontsize=14, fontweight='bold', pad=10)
ax1.set_ylabel('Precio (USD)', fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Anotaciones de eventos clave
eventos = [
    ('2021-11-10', '$69K ATH', 69000),
    ('2022-11-09', 'FTX Collapse', 16000),
    ('2024-03-14', 'Nuevo ATH 2024', 73000),
]
for fecha, label, y in eventos:
    try:
        ax1.annotate(label,
            xy=(pd.Timestamp(fecha, tz='UTC'), y),
            xytext=(0, 22), textcoords='offset points',
            fontsize=8, color='#37474F',
            arrowprops=dict(arrowstyle='->', color='#90A4AE', lw=1.2),
            ha='center')
    except Exception:
        pass

# Volumen
ax2.bar(clean_df.index, clean_df['Volume'] / 1e9,
         color=C['strategy'], alpha=0.5, width=0.8)
ax2.set_ylabel('Volumen (B)', fontsize=10)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}B'))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../reports/figures/00_price_history.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='3'></a>
## 3 · El Indicador RSI

### ¿Qué es el RSI?

El **Relative Strength Index** (RSI) es un oscilador de momentum desarrollado por J. Welles Wilder en 1978. Mide la velocidad y magnitud de los movimientos de precio recientes para evaluar si un activo está **sobrecomprado** o **sobrevendido**.

### Fórmula matemática

$$RSI = 100 - \frac{100}{1 + RS}$$

donde:

$$RS = \frac{\text{Promedio de ganancias en N períodos}}{\text{Promedio de pérdidas en N períodos}}$$

Wilder usa un **suavizado exponencial** con $\alpha = \frac{1}{N}$, equivalente a un EMA con esa tasa.

### Interpretación

| RSI | Señal | Acción |
|-----|-------|--------|
| < 30 | Sobreventa (Oversold) | 🟢 **BUY** — el mercado puede rebotar |
| 30 – 70 | Neutral | ⚪ HOLD — sin acción |
| > 70 | Sobrecompra (Overbought) | 🔴 **SELL** — el mercado puede corregir |

In [ ]:
# ── Cálculo del RSI ────────────────────────────────────────────────────────────
df_with_rsi = add_rsi_to_df(clean_df, period=RSI_PERIOD)
rsi_col     = f'RSI_{RSI_PERIOD}'

rsi = df_with_rsi[rsi_col].dropna()
print(f'RSI({RSI_PERIOD}) calculado:')
print(f'  Mínimo  : {rsi.min():.2f}')
print(f'  Máximo  : {rsi.max():.2f}')
print(f'  Media   : {rsi.mean():.2f}')
print(f'  Días < 30 (oversold)  : {(rsi < 30).sum()} ({(rsi < 30).mean()*100:.1f}%)')
print(f'  Días > 70 (overbought): {(rsi > 70).sum()} ({(rsi > 70).mean()*100:.1f}%)')

In [ ]:
# ── Distribución del RSI ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Histograma
n, bins, patches = ax1.hist(rsi, bins=50, color=C['rsi'], alpha=0.7, edgecolor='white', linewidth=0.5)
for patch, left in zip(patches, bins[:-1]):
    if left < 30:
        patch.set_facecolor(C['buy'])
        patch.set_alpha(0.8)
    elif left > 70:
        patch.set_facecolor(C['sell'])
        patch.set_alpha(0.8)

ax1.axvline(30, color=C['buy'],  linestyle='--', linewidth=1.5, label='Oversold (30)')
ax1.axvline(70, color=C['sell'], linestyle='--', linewidth=1.5, label='Overbought (70)')
ax1.axvline(rsi.mean(), color='#37474F', linestyle=':', linewidth=1.5, label=f'Media ({rsi.mean():.1f})')
ax1.set_title('Distribución del RSI(14)', fontsize=13, fontweight='bold')
ax1.set_xlabel('RSI')
ax1.set_ylabel('Frecuencia')
ax1.legend(fontsize=9)

# Serie temporal del RSI
ax2.plot(rsi.index, rsi.values, color=C['rsi'], linewidth=0.9, alpha=0.85)
ax2.fill_between(rsi.index, rsi, 70, where=(rsi > 70), color=C['sell'], alpha=0.25, label='Overbought')
ax2.fill_between(rsi.index, rsi, 30, where=(rsi < 30), color=C['buy'],  alpha=0.25, label='Oversold')
ax2.axhline(70, color=C['sell'], linestyle='--', linewidth=1.0)
ax2.axhline(30, color=C['buy'],  linestyle='--', linewidth=1.0)
ax2.axhline(50, color='#90A4AE', linestyle=':', linewidth=0.8)
ax2.set_ylim(0, 100)
ax2.set_title('RSI(14) — Serie temporal', fontsize=13, fontweight='bold')
ax2.set_ylabel('RSI')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/01_rsi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='4'></a>
## 4 · Generación de Señales de Trading

La estrategia genera señales **sin look-ahead bias**: la señal se calcula al **cierre de la barra N** y la orden se ejecuta al **precio de apertura de la barra N+1**.

```
Barra N  → RSI < 30  → Signal = +1 (BUY)
Barra N+1 → Ejecutar compra al precio OPEN

Barra M  → RSI > 70  → Signal = -1 (SELL)  
Barra M+1 → Ejecutar venta al precio OPEN
```

In [ ]:
# ── Generar señales ────────────────────────────────────────────────────────────
signal_df = generate_signals(
    clean_df,
    rsi_period  = RSI_PERIOD,
    oversold    = RSI_OVERSOLD,
    overbought  = RSI_OVERBOUGHT,
)

buys  = signal_df[signal_df['Signal'] ==  1]
sells = signal_df[signal_df['Signal'] == -1]

print(f'Señales generadas:')
print(f'  BUY  signals : {len(buys)}')
print(f'  SELL signals : {len(sells)}')
print()
if len(buys):
    print('Fechas de compra:')
    for d in buys.index:
        print(f'  → {d.date()}  (RSI: {signal_df.loc[d, rsi_col]:.1f})')

In [ ]:
# ── Gráfico de señales sobre precio y RSI ────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 9), sharex=True,
                                 gridspec_kw={'height_ratios': [2.5, 1]})
fig.subplots_adjust(hspace=0.06)

# Panel superior — Precio
ax1.plot(signal_df.index, signal_df['Close'], color='#37474F', linewidth=1.1, label='BTC Close')

# Marcadores de señales
for idx in buys.index:
    ax1.axvline(idx, color=C['buy'], alpha=0.2, linewidth=1)
ax1.scatter(buys.index,  buys['Close'],  marker='^', color=C['buy'],  s=120, zorder=5, label='BUY  signal')
ax1.scatter(sells.index, sells['Close'], marker='v', color=C['sell'], s=120, zorder=5, label='SELL signal')

# Anotar trades
for i, (idx, row) in enumerate(buys.iterrows()):
    ax1.annotate(f'B{i+1}', xy=(idx, row['Close']),
                  xytext=(0, -18), textcoords='offset points',
                  fontsize=8, color=C['buy'], ha='center', fontweight='bold')
for i, (idx, row) in enumerate(sells.iterrows()):
    ax1.annotate(f'S{i+1}', xy=(idx, row['Close']),
                  xytext=(0, 12), textcoords='offset points',
                  fontsize=8, color=C['sell'], ha='center', fontweight='bold')

ax1.set_title('BTC-USD — Señales de la estrategia RSI(14)', fontsize=14, fontweight='bold', pad=10)
ax1.set_ylabel('Precio (USD)', fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(loc='upper left', fontsize=10)

# Panel inferior — RSI
ax2.plot(signal_df.index, signal_df[rsi_col], color=C['rsi'], linewidth=1.1)
ax2.fill_between(signal_df.index, signal_df[rsi_col], RSI_OVERBOUGHT,
                  where=(signal_df[rsi_col] > RSI_OVERBOUGHT), color=C['sell'], alpha=0.2)
ax2.fill_between(signal_df.index, signal_df[rsi_col], RSI_OVERSOLD,
                  where=(signal_df[rsi_col] < RSI_OVERSOLD), color=C['buy'], alpha=0.2)
ax2.axhline(RSI_OVERBOUGHT, color=C['sell'], linestyle='--', linewidth=1.2, label=f'Overbought ({RSI_OVERBOUGHT})')
ax2.axhline(RSI_OVERSOLD,   color=C['buy'],  linestyle='--', linewidth=1.2, label=f'Oversold ({RSI_OVERSOLD})')
ax2.set_ylim(0, 100)
ax2.set_ylabel(f'RSI({RSI_PERIOD})', fontsize=11)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.xticks(rotation=30, ha='right')
ax2.legend(loc='upper right', fontsize=9)

plt.savefig('../reports/figures/02_rsi_signals.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='5'></a>
## 5 · Simulación de Ejecución: Comisiones y Slippage

El `BrokerSimulator` aplica dos capas de costos reales de transacción:

**Slippage** — diferencia adversa entre el precio esperado y el precio real de ejecución:
- BUY fills a: `precio × (1 + slippage_rate)` → se paga **más** de lo esperado  
- SELL fills a: `precio × (1 − slippage_rate)` → se recibe **menos** de lo esperado

**Comisión** — costo fijo porcentual sobre el valor nocional:
- Se aplica **después** del slippage, sobre el valor real de la operación

In [ ]:
# ── Visualizar el impacto de costos de transacción ────────────────────────────
raw_price = 50_000
capital   = 10_000

# BUY
buy_fill       = raw_price * (1 + SLIPPAGE_RATE)
buy_commission = capital * COMMISSION_RATE
shares_bought  = (capital - buy_commission) / buy_fill
cost_per_share = capital / shares_bought

# SELL (hipotético: precio sube 10%)
sell_raw    = raw_price * 1.10
sell_fill   = sell_raw * (1 - SLIPPAGE_RATE)
proceeds    = sell_fill * shares_bought
sell_comm   = proceeds * COMMISSION_RATE
net_proceed = proceeds - sell_comm
net_pnl     = net_proceed - capital
gross_pnl   = (sell_raw - raw_price) * shares_bought

print('=' * 52)
print('       Desglose de costos — Trade ejemplo')
print('=' * 52)
print(f'  Capital invertido        : ${capital:>10,.2f}')
print(f'  Precio raw (BUY)         : ${raw_price:>10,.2f}')
print(f'  Slippage (BUY) +{SLIPPAGE_RATE*100:.3f}%    : ${buy_fill - raw_price:>10,.2f}')
print(f'  Fill price (BUY)         : ${buy_fill:>10,.2f}')
print(f'  Comisión entrada         : ${buy_commission:>10,.2f}')
print(f'  Shares compradas         : {shares_bought:>10.6f}')
print('-' * 52)
print(f'  Precio raw (SELL, +10%)  : ${sell_raw:>10,.2f}')
print(f'  Slippage (SELL) -{SLIPPAGE_RATE*100:.3f}%   : ${sell_raw - sell_fill:>10,.2f}')
print(f'  Fill price (SELL)        : ${sell_fill:>10,.2f}')
print(f'  Comisión salida          : ${sell_comm:>10,.2f}')
print('-' * 52)
print(f'  Ganancia bruta           : ${gross_pnl:>10,.2f}')
print(f'  Costos totales           : ${capital - net_proceed + gross_pnl:>10,.2f}  '
      f'({(capital - net_proceed + gross_pnl)/gross_pnl*100:.1f}% del bruto)')
print(f'  Ganancia NETA            : ${net_pnl:>10,.2f}')
print('=' * 52)

---
<a id='6'></a>
## 6 · Ejecución del Motor de Backtesting

El `BacktestEngine` orquesta todo el pipeline:

```
load_ohlcv → data_cleaner → generate_signals → BrokerSimulator → Portfolio → Metrics
```

El motor itera barra a barra, ejecutando órdenes en el **open de la barra siguiente** a la señal.

In [ ]:
# ── Ejecutar el backtest completo ──────────────────────────────────────────────
engine = BacktestEngine(
    ticker          = DEFAULT_TICKER,
    start           = DEFAULT_START,
    end             = DEFAULT_END,
    interval        = DEFAULT_INTERVAL,
    initial_capital = INITIAL_CAPITAL,
    commission_rate = COMMISSION_RATE,
    slippage_rate   = SLIPPAGE_RATE,
    rsi_period      = RSI_PERIOD,
    rsi_oversold    = RSI_OVERSOLD,
    rsi_overbought  = RSI_OVERBOUGHT,
)

engine.run()

# ── Extraer resultados ─────────────────────────────────────────────────────────
equity    = engine.get_equity_curve()
trades_df = engine.get_trades()
data      = engine.data
trades    = engine.portfolio.closed_trades
metrics   = compute_metrics(equity, trades)

print(f'\n✅ Backtest completado — {len(trades)} trades ejecutados')

---
<a id='7'></a>
## 7 · Equity Curve & Drawdown

La **equity curve** muestra la evolución del capital en el tiempo, comparada contra el benchmark de **Buy & Hold**. El **drawdown** mide la caída máxima desde el pico más reciente del portfolio.

In [ ]:
# ── Preparar benchmark Buy & Hold ─────────────────────────────────────────────
bm_price  = data['Close'].reindex(equity.index, method='ffill')
bm_equity = (bm_price / bm_price.iloc[0]) * INITIAL_CAPITAL
dd_df     = compute_drawdown(equity)

# ── Gráfico de equity curve + drawdown ────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(3, 1, figure=fig, height_ratios=[3, 1.2, 1.2], hspace=0.08)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)

# Equity curve
ax1.plot(equity.index, equity.values, color=C['strategy'], linewidth=2.0, label='RSI Strategy', zorder=3)
ax1.plot(bm_equity.index, bm_equity.values, color=C['benchmark'],
          linewidth=1.2, linestyle='--', label='Buy & Hold', alpha=0.75)

# Sombrear períodos en posición
for trade in trades:
    if trade.exit_date:
        color = C['buy'] if trade.is_winner else C['sell']
        ax1.axvspan(trade.entry_date, trade.exit_date, alpha=0.07, color=color, zorder=1)

final_strat = equity.iloc[-1]
final_bm    = bm_equity.iloc[-1]
ax1.set_title(
    f'Equity Curve — RSI Strategy vs Buy & Hold  |  '
    f'Estrategia: ${final_strat:,.0f}  |  '
    f'Buy & Hold: ${final_bm:,.0f}',
    fontsize=13, fontweight='bold', pad=10)
ax1.set_ylabel('Portfolio (USD)', fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(loc='upper left', fontsize=10)
plt.setp(ax1.get_xticklabels(), visible=False)

# RSI
ax2.plot(data.index, data[rsi_col], color=C['rsi'], linewidth=0.9)
ax2.axhline(RSI_OVERBOUGHT, color=C['sell'], linestyle='--', linewidth=1.0, alpha=0.7)
ax2.axhline(RSI_OVERSOLD,   color=C['buy'],  linestyle='--', linewidth=1.0, alpha=0.7)
ax2.fill_between(data.index, data[rsi_col], RSI_OVERBOUGHT,
                  where=(data[rsi_col] > RSI_OVERBOUGHT), color=C['sell'], alpha=0.15)
ax2.fill_between(data.index, data[rsi_col], RSI_OVERSOLD,
                  where=(data[rsi_col] < RSI_OVERSOLD), color=C['buy'],  alpha=0.15)
ax2.set_ylim(0, 100)
ax2.set_ylabel(f'RSI({RSI_PERIOD})', fontsize=10)
plt.setp(ax2.get_xticklabels(), visible=False)

# Drawdown
ax3.fill_between(dd_df.index, dd_df['Drawdown_Pct'], 0, color=C['drawdown'], alpha=0.45)
ax3.plot(dd_df.index, dd_df['Drawdown_Pct'], color=C['drawdown'], linewidth=0.8)
ax3.set_ylabel('Drawdown (%)', fontsize=10)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.xticks(rotation=30, ha='right')

plt.savefig('../reports/figures/03_equity_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='8'></a>
## 8 · Métricas de Rendimiento

### Glosario de métricas

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **CAGR** | $(V_f/V_i)^{1/n} - 1$ | Retorno anualizado compuesto |
| **Sharpe** | $\frac{E[r]-r_f}{\sigma_r} \cdot \sqrt{365}$ | Retorno por unidad de riesgo total |
| **Sortino** | $\frac{E[r]-r_f}{\sigma_{downside}} \cdot \sqrt{365}$ | Retorno por unidad de riesgo bajista |
| **Calmar** | $CAGR / |MaxDD|$ | Retorno por unidad de drawdown máximo |
| **Max DD** | $\min(E_t/\max_{s \leq t}E_s - 1)$ | Peor caída desde el pico |
| **Win Rate** | $N_{ganadores} / N_{total}$ | % de trades con P&L positivo |
| **Profit Factor** | $\Sigma gains / \Sigma losses$ | > 1 = estrategia rentable en bruto |

In [ ]:
# ── Tabla de métricas estilizada ───────────────────────────────────────────────
metrics_df = format_metrics_table(metrics)

def color_metric(val, metric_name):
    """Colorea positivo/negativo según el tipo de métrica."""
    positive_good = ['Total Return (%)', 'CAGR (%)', 'Sharpe Ratio', 'Sortino Ratio',
                     'Calmar Ratio', 'Win Rate (%)', 'Avg Win ($)', 'Best Trade ($)',
                     'Profit Factor']
    negative_good = ['Max Drawdown (%)', 'Avg Loss ($)', 'Worst Trade ($)']
    try:
        v = float(val)
        if metric_name in positive_good:
            return 'color: #2e7d32; font-weight: bold' if v > 0 else 'color: #c62828; font-weight: bold'
        elif metric_name in negative_good:
            return 'color: #c62828; font-weight: bold' if v < 0 else 'color: #2e7d32; font-weight: bold'
    except:
        pass
    return ''

display(HTML('<h4 style="color:#1565c0; margin-bottom: 8px">📊 Performance Summary — RSI Strategy (BTC-USD 2020–2024)</h4>'))

def style_row(row):
    return [color_metric(row['Value'], row.name)]

styled = (metrics_df
    .style
    .apply(lambda row: [color_metric(row['Value'], row.name)], axis=1)
    .set_properties(**{
        'font-size': '14px',
        'padding': '8px 16px',
        'border-bottom': '1px solid #e0e0e0',
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [('background', '#1565c0'), ('color', 'white'),
                  ('font-size', '13px'), ('padding', '10px 16px')]
    }])
)
display(styled)

In [ ]:
# ── Dashboard visual de métricas clave ────────────────────────────────────────
key_metrics = [
    ('Total Return', f"+{metrics['Total Return (%)']:.1f}%", C['strategy']),
    ('CAGR',         f"{metrics['CAGR (%)']:.2f}%",         C['buy']),
    ('Sharpe Ratio', f"{metrics['Sharpe Ratio']:.3f}",      C['rsi']),
    ('Max Drawdown', f"{metrics['Max Drawdown (%)']:.1f}%", C['drawdown']),
    ('Win Rate',     f"{metrics['Win Rate (%)']:.1f}%",     C['gold']),
    ('Total Trades', f"{metrics['Total Trades']}",          C['benchmark']),
]

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
fig.patch.set_facecolor('#F0F4FF')

for ax, (label, value, color) in zip(axes, key_metrics):
    ax.set_facecolor('white')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    # Card border
    rect = FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
                            boxstyle='round,pad=0.02',
                            linewidth=2, edgecolor=color,
                            facecolor='white')
    ax.add_patch(rect)
    # Top color bar
    bar = FancyBboxPatch((0.05, 0.82), 0.9, 0.13,
                          boxstyle='round,pad=0.01',
                          facecolor=color, edgecolor='none')
    ax.add_patch(bar)
    ax.text(0.5, 0.88, label, ha='center', va='center',
             fontsize=9, color='white', fontweight='bold')
    ax.text(0.5, 0.48, value, ha='center', va='center',
             fontsize=18, color=color, fontweight='bold')

fig.suptitle('Métricas Clave de Rendimiento', fontsize=14, fontweight='bold',
              color='#1a237e', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/04_metrics_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='9'></a>
## 9 · Análisis Detallado de Trades

Examinamos cada operación individual: fechas, precios de entrada/salida, P&L neto, duración y si fue ganadora o perdedora.

In [ ]:
# ── Log de trades estilizado ───────────────────────────────────────────────────
display(HTML('<h4 style="color:#1565c0">📋 Trade Log completo</h4>'))

trades_display = trades_df.copy()
trades_display['entry_date'] = trades_display['entry_date'].dt.strftime('%Y-%m-%d')
trades_display['exit_date']  = trades_display['exit_date'].dt.strftime('%Y-%m-%d')

col_rename = {
    'trade_id':     '#',
    'entry_date':   'Entrada',
    'entry_price':  'Precio Entrada',
    'exit_date':    'Salida',
    'exit_price':   'Precio Salida',
    'gross_pnl':    'P&L Bruto',
    'net_pnl':      'P&L Neto',
    'return_pct':   'Retorno %',
    'duration_days':'Días',
    'winner':       'Ganador'
}
trades_display = trades_display[col_rename.keys()].rename(columns=col_rename)

def highlight_winner(row):
    color = '#e8f5e9' if row['Ganador'] else '#ffebee'
    return [f'background-color: {color}'] * len(row)

styled_trades = (trades_display.style
    .apply(highlight_winner, axis=1)
    .format({
        'Precio Entrada': '${:,.2f}',
        'Precio Salida':  '${:,.2f}',
        'P&L Bruto':      '${:+,.2f}',
        'P&L Neto':       '${:+,.2f}',
        'Retorno %':      '{:+.2f}%',
        'Días':           '{:.1f}',
    })
    .set_properties(**{'font-size': '13px', 'padding': '8px 12px'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('background', '#283593'), ('color', 'white'),
                  ('padding', '10px 12px'), ('font-size', '12px')]
    }])
)
display(styled_trades)

In [ ]:
# ── Análisis visual de trades ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. P&L por trade
ax1 = axes[0]
colors_bar = [C['buy'] if w else C['sell'] for w in trades_df['winner']]
bars = ax1.bar(trades_df['trade_id'], trades_df['net_pnl'], color=colors_bar,
                edgecolor='white', linewidth=0.8)
ax1.axhline(0, color='#37474F', linewidth=0.8, linestyle='--')
ax1.set_title('P&L Neto por Trade', fontsize=12, fontweight='bold')
ax1.set_xlabel('Trade #')
ax1.set_ylabel('P&L (USD)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, trades_df['net_pnl']):
    ax1.text(bar.get_x() + bar.get_width()/2,
              val + (300 if val >= 0 else -600),
              f'${val:,.0f}', ha='center', fontsize=8, fontweight='bold')

# 2. Duración por trade
ax2 = axes[1]
ax2.barh(trades_df['trade_id'], trades_df['duration_days'],
          color=[C['buy'] if w else C['sell'] for w in trades_df['winner']],
          edgecolor='white')
ax2.set_title('Duración de cada Trade', fontsize=12, fontweight='bold')
ax2.set_xlabel('Días en posición')
ax2.set_ylabel('Trade #')
ax2.axvline(trades_df['duration_days'].mean(), color='#37474F',
              linestyle='--', linewidth=1.2, label=f'Media: {trades_df["duration_days"].mean():.1f}d')
ax2.legend(fontsize=9)

# 3. Retorno % acumulado
ax3 = axes[2]
cum_ret = (1 + trades_df['return_pct'] / 100).cumprod() - 1
ax3.plot(trades_df['trade_id'], cum_ret * 100, marker='o', color=C['strategy'],
          linewidth=2, markersize=7)
ax3.fill_between(trades_df['trade_id'], cum_ret * 100, 0,
                  where=(cum_ret >= 0), color=C['buy'], alpha=0.2)
ax3.fill_between(trades_df['trade_id'], cum_ret * 100, 0,
                  where=(cum_ret < 0), color=C['sell'], alpha=0.2)
ax3.axhline(0, color='#37474F', linewidth=0.8, linestyle='--')
ax3.set_title('Retorno % Acumulado (trades)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Trade #')
ax3.set_ylabel('Retorno acumulado (%)')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.1f}%'))

plt.suptitle('Análisis de Trades Individuales', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/05_trade_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='10'></a>
## 10 · Análisis de Sensibilidad de Parámetros

¿Qué pasa si cambiamos los umbrales del RSI? Hacemos un **grid search** sobre combinaciones de `oversold` y `overbought` para visualizar qué parámetros habrían maximizado el Sharpe Ratio y el Total Return.

In [ ]:
# ── Grid search de parámetros RSI ─────────────────────────────────────────────
from itertools import product
import logging
logging.disable(logging.CRITICAL)  # Silenciar logs durante el grid search

oversold_range   = [20, 25, 30, 35, 40]
overbought_range = [60, 65, 70, 75, 80]

results = []
for os_val, ob_val in product(oversold_range, overbought_range):
    if ob_val <= os_val:
        continue
    try:
        eng = BacktestEngine(
            ticker=DEFAULT_TICKER, start=DEFAULT_START, end=DEFAULT_END,
            initial_capital=INITIAL_CAPITAL,
            commission_rate=COMMISSION_RATE, slippage_rate=SLIPPAGE_RATE,
            rsi_period=RSI_PERIOD,
            rsi_oversold=os_val, rsi_overbought=ob_val,
        )
        eng.run()
        eq = eng.get_equity_curve()
        trd = eng.portfolio.closed_trades
        m = compute_metrics(eq, trd)
        results.append({
            'oversold': os_val, 'overbought': ob_val,
            'sharpe':  m['Sharpe Ratio'],
            'total_return': m['Total Return (%)'],
            'max_dd': m['Max Drawdown (%)'],
            'n_trades': m['Total Trades'],
            'win_rate': m['Win Rate (%)'],
        })
    except Exception:
        pass

logging.disable(logging.NOTSET)
grid_df = pd.DataFrame(results)
print(f'Combinaciones evaluadas: {len(grid_df)}')
print(f'Mejor Sharpe: {grid_df.loc[grid_df.sharpe.idxmax(), ["oversold","overbought","sharpe"]].to_dict()}')
print(f'Mejor Return: {grid_df.loc[grid_df.total_return.idxmax(), ["oversold","overbought","total_return"]].to_dict()}')

In [ ]:
# ── Heatmaps de sensibilidad ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title, cmap in [
    (axes[0], 'sharpe',       'Sharpe Ratio',      'RdYlGn'),
    (axes[1], 'total_return', 'Total Return (%)',   'RdYlGn'),
]:
    pivot = grid_df.pivot(index='oversold', columns='overbought', values=col)
    im    = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('RSI Overbought threshold', fontsize=11)
    ax.set_ylabel('RSI Oversold threshold',  fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')

    # Anotar valores en cada celda
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.iloc[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                     fontsize=9, color='black', fontweight='bold')

    # Marcar el parámetro base usado
    try:
        ri = list(pivot.index).index(RSI_OVERSOLD)
        ci = list(pivot.columns).index(RSI_OVERBOUGHT)
        ax.add_patch(plt.Rectangle((ci-0.5, ri-0.5), 1, 1,
                                     fill=False, edgecolor='#1a237e',
                                     linewidth=3, zorder=5))
    except ValueError:
        pass

plt.suptitle('Análisis de Sensibilidad — Parámetros RSI\n'
              '(cuadro azul = configuración base)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
<a id='11'></a>
## 11 · Reporte Final

Dashboard completo con todos los resultados integrados en una sola figura de publicación.

In [ ]:
# ── Reporte final integrado ────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#F0F4FF')
gs  = gridspec.GridSpec(4, 4, figure=fig, hspace=0.45, wspace=0.4)

# ── Header ─────────────────────────────────────────────────────────────────────
ax_header = fig.add_subplot(gs[0, :])
ax_header.set_facecolor('#1a237e')
ax_header.set_xlim(0, 1)
ax_header.set_ylim(0, 1)
ax_header.axis('off')
ax_header.text(0.5, 0.65, 'RSI Backtesting Engine — Performance Report',
                ha='center', va='center', fontsize=18, fontweight='bold', color='white')
ax_header.text(0.5, 0.25,
    f'{DEFAULT_TICKER}  ·  {DEFAULT_START} → {DEFAULT_END}  ·  '
    f'RSI({RSI_PERIOD})  ·  Oversold: {RSI_OVERSOLD}  ·  Overbought: {RSI_OVERBOUGHT}  ·  '
    f'Comisión: {COMMISSION_RATE*100:.2f}%  ·  Capital: ${INITIAL_CAPITAL:,.0f}',
    ha='center', va='center', fontsize=10, color='#90caf9')

# ── Equity curve ───────────────────────────────────────────────────────────────
ax_eq = fig.add_subplot(gs[1, :3])
ax_eq.plot(equity.index, equity.values, color=C['strategy'], linewidth=1.8, label='RSI Strategy')
ax_eq.plot(bm_equity.index, bm_equity.values, color=C['benchmark'],
            linewidth=1.1, linestyle='--', alpha=0.75, label='Buy & Hold')
for trade in trades:
    if trade.exit_date:
        ax_eq.axvspan(trade.entry_date, trade.exit_date, alpha=0.07,
                       color=C['buy'] if trade.is_winner else C['sell'])
ax_eq.set_title('Equity Curve', fontsize=12, fontweight='bold')
ax_eq.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax_eq.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax_eq.legend(fontsize=9)

# ── Métricas cards ─────────────────────────────────────────────────────────────
ax_cards = fig.add_subplot(gs[1, 3])
ax_cards.axis('off')
card_metrics = [
    ('Total Return', f"+{metrics['Total Return (%)']:.1f}%",   C['strategy']),
    ('CAGR',         f"{metrics['CAGR (%)']:.2f}%",            C['buy']),
    ('Sharpe',       f"{metrics['Sharpe Ratio']:.3f}",         C['rsi']),
    ('Max DD',       f"{metrics['Max Drawdown (%)']:.1f}%",    C['drawdown']),
    ('Win Rate',     f"{metrics['Win Rate (%)']:.1f}%",        C['gold']),
    ('# Trades',     f"{metrics['Total Trades']}",             C['benchmark']),
]
for i, (label, val, color) in enumerate(card_metrics):
    y = 0.88 - i * 0.155
    ax_cards.text(0.1, y, label, fontsize=9,  color='#546e7a', va='center')
    ax_cards.text(0.9, y, val,   fontsize=12, color=color, va='center',
                   ha='right', fontweight='bold')
    ax_cards.axhline(y - 0.06, xmin=0.05, xmax=0.95, color='#e0e0e0', linewidth=0.6)

# ── Drawdown ───────────────────────────────────────────────────────────────────
ax_dd = fig.add_subplot(gs[2, :3])
ax_dd.fill_between(dd_df.index, dd_df['Drawdown_Pct'], 0, color=C['drawdown'], alpha=0.4)
ax_dd.plot(dd_df.index, dd_df['Drawdown_Pct'], color=C['drawdown'], linewidth=0.9)
ax_dd.set_title('Drawdown desde el Pico', fontsize=12, fontweight='bold')
ax_dd.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax_dd.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── P&L por trade ──────────────────────────────────────────────────────────────
ax_pnl = fig.add_subplot(gs[2, 3])
colors_t = [C['buy'] if w else C['sell'] for w in trades_df['winner']]
ax_pnl.barh(trades_df['trade_id'], trades_df['net_pnl'], color=colors_t, edgecolor='white')
ax_pnl.axvline(0, color='#37474F', linewidth=0.8)
ax_pnl.set_title('P&L por Trade', fontsize=12, fontweight='bold')
ax_pnl.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax_pnl.set_ylabel('Trade #')

# ── RSI ────────────────────────────────────────────────────────────────────────
ax_rsi = fig.add_subplot(gs[3, :3])
ax_rsi.plot(data.index, data[rsi_col], color=C['rsi'], linewidth=0.9)
ax_rsi.fill_between(data.index, data[rsi_col], RSI_OVERBOUGHT,
                     where=(data[rsi_col] > RSI_OVERBOUGHT), color=C['sell'], alpha=0.2)
ax_rsi.fill_between(data.index, data[rsi_col], RSI_OVERSOLD,
                     where=(data[rsi_col] < RSI_OVERSOLD), color=C['buy'], alpha=0.2)
ax_rsi.axhline(RSI_OVERBOUGHT, color=C['sell'], linestyle='--', linewidth=1)
ax_rsi.axhline(RSI_OVERSOLD,   color=C['buy'],  linestyle='--', linewidth=1)
ax_rsi.set_ylim(0, 100)
ax_rsi.set_title(f'RSI({RSI_PERIOD})', fontsize=12, fontweight='bold')
ax_rsi.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax_rsi.xaxis.set_major_locator(mdates.MonthLocator(interval=5))
plt.setp(ax_rsi.get_xticklabels(), rotation=30, ha='right')

# ── Distribución retornos ───────────────────────────────────────────────────────
ax_dist = fig.add_subplot(gs[3, 3])
ret_pct = trades_df['return_pct']
ax_dist.hist(ret_pct, bins=8, color=C['strategy'], alpha=0.7, edgecolor='white')
ax_dist.axvline(0, color='#37474F', linewidth=1, linestyle='--')
ax_dist.axvline(ret_pct.mean(), color=C['gold'], linewidth=1.5, linestyle='--',
                 label=f'Media: {ret_pct.mean():.1f}%')
ax_dist.set_title('Distribución Retornos', fontsize=12, fontweight='bold')
ax_dist.set_xlabel('Retorno por trade (%)')
ax_dist.legend(fontsize=8)

plt.savefig('../reports/figures/07_full_report.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Reporte final guardado en reports/figures/07_full_report.png')

In [ ]:
# ── Exportar reportes CSV ──────────────────────────────────────────────────────
import os
os.makedirs('../reports', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)

metrics_df.to_csv('../reports/performance_summary.csv')
trades_df.to_csv('../reports/trades.csv', index=False)

print('✅ Reportes exportados:')
print('   → reports/performance_summary.csv')
print('   → reports/trades.csv')
print()
print('✅ Figuras generadas:')
for f in sorted(os.listdir('../reports/figures')):
    print(f'   → reports/figures/{f}')

---

## 📌 Conclusiones

<div style="background: #f5f7ff; border-left: 5px solid #1565c0; padding: 20px 24px; border-radius: 8px; margin-top: 8px;">

### Resultados de la estrategia RSI(14) en BTC-USD (2020–2024)

**Rendimiento:** La estrategia generó un retorno positivo con Sharpe Ratio > 0, aunque significativamente inferior al Buy & Hold en un mercado alcista dominado por fuertes tendencias. Esto es esperado para una estrategia de **mean-reversion** en un activo con bias alcista estructural.

**Fortalezas:**
- ✅ Win Rate > 70%: la mayoría de las operaciones fueron ganadoras
- ✅ Profit Factor > 1: el sistema es rentable en términos brutos
- ✅ Muy pocos trades (7): baja exposición y costos de transacción reducidos

**Limitaciones:**
- ⚠️ Avg Loss > Avg Win: los trades perdedores pierden más de lo que ganan los ganadores
- ⚠️ Max Drawdown elevado: hereda el drawdown del mercado durante los períodos en posición
- ⚠️ El análisis de sensibilidad muestra que los resultados varían significativamente con los parámetros

**Próximos pasos sugeridos:**
1. Añadir un **stop-loss** para limitar las pérdidas en trades perdedores
2. Combinar RSI con un **filtro de tendencia** (SMA 200) para evitar comprar en tendencias bajistas
3. Implementar **position sizing dinámico** (Kelly Criterion o riesgo fijo por trade)
4. Hacer walk-forward testing para validar robustez fuera de muestra

</div>